In [33]:
import pandas as pd 
import pyodbc
import sqlalchemy as sal
from sqlalchemy import text
from sqlalchemy import create_engine
import logging 
import os
import time 
from datetime import datetime

In [4]:
engine = sal.create_engine(r'mssql://ABHISHEK\SQLEXPRESS/Hospital_db?driver=ODBC+DRIVER+17+FOR+SQL+SERVER')
conn=engine.connect() # For connnecting your SQL Database with Python with Hospital_db database

In [13]:
query = "SELECT * FROM patients"
patients_df = pd.read_sql(query, engine)

In [29]:
query = "SELECT * FROM appointment"
appointment_df = pd.read_sql(query, engine)

In [15]:
query = "SELECT * FROM doctors"
doctors_df = pd.read_sql(query, engine)

In [16]:
query = "SELECT * FROM billing"
billing_df = pd.read_sql(query, engine)

In [17]:
query = "SELECT * FROM treatments"
treatments_df = pd.read_sql(query, engine)

## KPI's 

In [30]:
# Total Patients
total_patients = patients_df['patient_id'].nunique()

# Total Doctors
total_doctors = doctors_df['doctor_id'].nunique()

# Total Appointments
total_appointments = appointment_df['appointment_id'].nunique()

# Total Treatments
total_treatments = treatments_df['treatment_id'].nunique()

# Total Revenue
total_revenue = billing_df['amount'].sum()

# Average Bill Amount
average_bill = billing_df['amount'].mean()

In [31]:
print(f"Total Patients      : {total_patients}")
print(f"Total Doctors       : {total_doctors}")
print(f"Total Appointments  : {total_appointments}")
print(f"Total Treatments    : {total_treatments}")
print(f"Total Revenue       : ₹{total_revenue:,.2f}")
print(f"Average Bill Amount : ₹{average_bill:,.2f}")

Total Patients      : 50
Total Doctors       : 10
Total Appointments  : 200
Total Treatments    : 200
Total Revenue       : ₹551,249.85
Average Bill Amount : ₹2,756.25


## Gender Distribution

In [32]:
gender_distribution = (
    patients_df['gender']
    .value_counts()
    .reset_index()
)

gender_distribution.columns = ['Gender', 'Count']

gender_distribution

,Gender,Count
0,M,31
1,F,19


## Age Calculation

In [36]:
today = pd.Timestamp.today()

patients_df['age'] = (
    (today - patients_df['date_of_birth']).dt.days // 365
)

patients_df[['patient_id', 'date_of_birth', 'age']].head(5)

,patient_id,date_of_birth,age
0,P001,1955-06-04,71
1,P002,1984-10-12,41
2,P003,1977-08-21,48
3,P004,1981-02-20,45
4,P005,1960-06-23,66


## Age Group Distribution

In [37]:
patients_df['age_group'] = pd.cut(
    patients_df['age'],
    bins=[0, 18, 30, 45, 60, 100],
    labels=['0-18', '19-30', '31-45', '46-60', '60+'],
    include_lowest=True
)

age_group_distribution = (
    patients_df['age_group']
    .value_counts()
    .sort_index()
    .reset_index()
)

age_group_distribution.columns = ['Age Group', 'Count']

age_group_distribution

,Age Group,Count
0,0-18,0
1,19-30,10
2,31-45,16
3,46-60,14
4,60+,10


## the Registration Trend

In [38]:
registration_trend = (
    patients_df
    .groupby(patients_df['registration_date'].dt.to_period('M'))
    .size()
    .reset_index(name='Registrations')
)

registration_trend['registration_date'] = registration_trend['registration_date'].astype(str)

registration_trend

,registration_date,Registrations
0,2021-01,1
1,2021-03,2
2,2021-04,1
3,2021-05,3
4,2021-07,2
5,2021-08,2
6,2021-09,5
7,2021-10,2
8,2021-12,3
9,2022-01,2


## Doctor Specialization Distribution.

In [39]:
specialization_distribution = (
    doctors_df['specialization']
    .value_counts()
    .reset_index()
)

specialization_distribution.columns = ['Specialization', 'Count']

specialization_distribution

,Specialization,Count
0,Pediatrics,5
1,Dermatology,3
2,Oncology,2


## Doctor Distribution by Hospital Branch.

In [40]:
branch_distribution = (
    doctors_df['hospital_branch']
    .value_counts()
    .reset_index()
)

branch_distribution.columns = ['Hospital Branch', 'Count']

branch_distribution

,Hospital Branch,Count
0,Central Hospital,4
1,Westside Clinic,3
2,Eastside Clinic,3


## Appointment Status Distribution.

In [41]:
appointment_status = (
    appointment_df['status']
    .value_counts()
    .reset_index()
)

appointment_status.columns = ['Status', 'Count']

appointment_status

,Status,Count
0,No-show,52
1,Scheduled,51
2,Cancelled,51
3,Completed,46


## Monthly Appointment Trend.

In [42]:
monthly_appointments = (
    appointment_df
    .groupby(appointment_df['appointment_date'].dt.to_period('M'))
    .size()
    .reset_index(name='Appointments')
)

monthly_appointments['appointment_date'] = monthly_appointments['appointment_date'].astype(str)

monthly_appointments

,appointment_date,Appointments
0,2023-01,20
1,2023-02,14
2,2023-03,19
3,2023-04,25
4,2023-05,19
5,2023-06,18
6,2023-07,16
7,2023-08,15
8,2023-09,11
9,2023-10,14


## Treatment Type Distribution.

In [43]:
treatment_distribution = (
    treatments_df['treatment_type']
    .value_counts()
    .reset_index()
)

treatment_distribution.columns = ['Treatment Type', 'Count']

treatment_distribution

,Treatment Type,Count
0,Chemotherapy,49
1,X-Ray,41
2,ECG,38
3,MRI,36
4,Physiotherapy,36


## Payment Method Distribution.

In [44]:
payment_method_distribution = (
    billing_df['payment_method']
    .value_counts()
    .reset_index()
)

payment_method_distribution.columns = ['Payment Method', 'Count']

payment_method_distribution

,Payment Method,Count
0,Credit Card,75
1,Insurance,64
2,Cash,61


## Payment Status Distribution.

In [45]:
payment_status_distribution = (
    billing_df['payment_status']
    .value_counts()
    .reset_index()
)

payment_status_distribution.columns = ['Payment Status', 'Count']

payment_status_distribution

,Payment Status,Count
0,Pending,69
1,Failed,67
2,Paid,64


## Monthly Revenue Trend.

In [46]:
monthly_revenue = (
    billing_df
    .groupby(billing_df['bill_date'].dt.to_period('M'))['amount']
    .sum()
    .reset_index()
)

monthly_revenue['bill_date'] = monthly_revenue['bill_date'].astype(str)

monthly_revenue

,bill_date,amount
0,2023-01,58701.23
1,2023-02,36669.69
2,2023-03,47304.29
3,2023-04,64271.54
4,2023-05,48791.05
5,2023-06,56887.82
6,2023-07,39880.19
7,2023-08,41958.67
8,2023-09,33426.53
9,2023-10,43314.15


## Average Treatment Cost by Treatment Type.

In [47]:
average_treatment_cost = (
    treatments_df
    .groupby('treatment_type')['cost']
    .mean()
    .reset_index()
)

average_treatment_cost.columns = ['Treatment Type', 'Average Cost']

average_treatment_cost

,Treatment Type,Average Cost
0,Chemotherapy,2629.707755
1,ECG,2532.216842
2,MRI,3224.948889
3,Physiotherapy,2761.613889
4,X-Ray,2698.870000


## Top 10 Highest Billing Amounts.

In [49]:
top_10_bills = (
    billing_df
    .sort_values(by='amount', ascending=False)
    .head(10)
)

In [50]:
paid_bills = billing_df[
    billing_df['payment_status'] == 'Paid'
]

In [51]:
top_10_paid_bills = (
    paid_bills
    .sort_values(by='amount', ascending=False)
    .head(10)
)

top_10_paid_bills

,bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status
107,B108,P024,T108,2023-04-21,4973.63,Cash,Paid
191,B192,P038,T192,2023-08-31,4846.20,Insurance,Paid
189,B190,P029,T190,2023-11-16,4834.02,Credit Card,Paid
114,B115,P049,T115,2023-10-25,4809.31,Insurance,Paid
12,B013,P003,T013,2023-08-16,4704.96,Cash,Paid
8,B009,P039,T009,2023-03-05,4541.14,Credit Card,Paid
90,B091,P010,T091,2023-06-11,4523.86,Credit Card,Paid
44,B045,P010,T045,2023-09-28,4478.93,Cash,Paid
132,B133,P048,T133,2023-03-23,4289.15,Insurance,Paid
55,B056,P049,T056,2023-01-02,4201.76,Insurance,Paid


## Total Revenue from Paid Bills.

In [53]:
paid_revenue = (
    billing_df[billing_df['payment_status'] == 'Paid']['amount']
    .sum()
)

print(f"Total Paid Revenue: ₹{paid_revenue:,.2f}")

Total Paid Revenue: ₹173,424.90


## Revenue by Payment Method (using only paid bills).

In [54]:
revenue_by_payment_method = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .groupby('payment_method')['amount']
    .sum()
    .reset_index()
)

revenue_by_payment_method.columns = ['Payment Method', 'Revenue']

revenue_by_payment_method

,Payment Method,Revenue
0,Cash,52691.30
1,Credit Card,60377.11
2,Insurance,60356.49


## Revenue by Treatment Type (using only paid bills).

In [55]:
revenue_by_treatment = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .merge(
        treatments_df[['treatment_id', 'treatment_type']],
        on='treatment_id',
        how='left'
    )
    .groupby('treatment_type')['amount']
    .sum()
    .reset_index()
    .sort_values(by='amount', ascending=False)
)

revenue_by_treatment.columns = ['Treatment Type', 'Revenue']

revenue_by_treatment

,Treatment Type,Revenue
4,X-Ray,47978.78
2,MRI,43064.42
0,Chemotherapy,32607.26
3,Physiotherapy,32251.38
1,ECG,17523.06


## Top 10 Doctors by Number of Appointments.

In [56]:
top_doctors = (
    appointment_df
    .merge(
        doctors_df[['doctor_id', 'first_name', 'last_name']],
        on='doctor_id',
        how='left'
    )
    .assign(doctor_name=lambda x: x['first_name'] + ' ' + x['last_name'])
    .groupby('doctor_name')
    .size()
    .reset_index(name='Total Appointments')
    .sort_values(by='Total Appointments', ascending=False)
    .head(10)
)

top_doctors

,doctor_name,Total Appointments
9,Sarah Taylor,29
2,David Taylor,25
0,Alex Davis,24
4,Jane Smith,22
3,Jane Davis,21
6,Linda Wilson,19
8,Sarah Smith,17
5,Linda Brown,16
1,David Jones,14
7,Robert Davis,13


## Top 10 Patients by Total Billing Amount

In [57]:
top_patients = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .merge(
        patients_df[['patient_id', 'first_name', 'last_name']],
        on='patient_id',
        how='left'
    )
    .assign(patient_name=lambda x: x['first_name'] + ' ' + x['last_name'])
    .groupby('patient_name')['amount']
    .sum()
    .reset_index()
    .sort_values(by='amount', ascending=False)
    .head(10)
)

top_patients.columns = ['Patient Name', 'Total Billing']

top_patients

,Patient Name,Total Billing
20,Michael Taylor,20801.97
14,Laura Davis,19831.04
6,David Wilson,15369.27
3,David Moore,12149.85
7,Emily Miller,11711.41
1,Alex Moore,11105.32
4,David Smith,10794.20
0,Alex Johnson,8636.90
24,Sarah Brown,7506.58
22,Robert Miller,7178.46


## Top 10 Doctors by Revenue

In [58]:
top_doctors_revenue = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .merge(
        treatments_df[['treatment_id', 'appointment_id']],
        on='treatment_id',
        how='left'
    )
    .merge(
        appointment_df[['appointment_id', 'doctor_id']],
        on='appointment_id',
        how='left'
    )
    .merge(
        doctors_df[['doctor_id', 'first_name', 'last_name']],
        on='doctor_id',
        how='left'
    )
    .assign(doctor_name=lambda x: x['first_name'] + ' ' + x['last_name'])
    .groupby('doctor_name')['amount']
    .sum()
    .reset_index()
    .sort_values(by='amount', ascending=False)
    .head(10)
)

top_doctors_revenue.columns = ['Doctor Name', 'Revenue']

top_doctors_revenue

,Doctor Name,Revenue
9,Sarah Taylor,33836.91
0,Alex Davis,25698.80
6,Linda Wilson,18935.50
2,David Taylor,18572.75
3,Jane Davis,18201.70
1,David Jones,17693.42
4,Jane Smith,17260.25
5,Linda Brown,12919.88
8,Sarah Smith,9230.98
7,Robert Davis,1074.71


## Monthly Revenue by Hospital Branch

In [59]:
revenue_by_branch = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .merge(
        treatments_df[['treatment_id', 'appointment_id']],
        on='treatment_id',
        how='left'
    )
    .merge(
        appointment_df[['appointment_id', 'doctor_id']],
        on='appointment_id',
        how='left'
    )
    .merge(
        doctors_df[['doctor_id', 'hospital_branch']],
        on='doctor_id',
        how='left'
    )
    .groupby('hospital_branch')['amount']
    .sum()
    .reset_index()
    .sort_values(by='amount', ascending=False)
)

revenue_by_branch.columns = ['Hospital Branch', 'Revenue']

revenue_by_branch

,Hospital Branch,Revenue
0,Central Hospital,86460.11
1,Eastside Clinic,54397.45
2,Westside Clinic,32567.34


## Revenue by Doctor Specialization

In [60]:
revenue_by_specialization = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .merge(
        treatments_df[['treatment_id', 'appointment_id']],
        on='treatment_id',
        how='left'
    )
    .merge(
        appointment_df[['appointment_id', 'doctor_id']],
        on='appointment_id',
        how='left'
    )
    .merge(
        doctors_df[['doctor_id', 'specialization']],
        on='doctor_id',
        how='left'
    )
    .groupby('specialization')['amount']
    .sum()
    .reset_index()
    .sort_values(by='amount', ascending=False)
)

revenue_by_specialization.columns = ['Specialization', 'Revenue']

revenue_by_specialization

,Specialization,Revenue
2,Pediatrics,88085.15
0,Dermatology,65329.54
1,Oncology,20010.21


## Average Revenue per Patient

In [61]:
average_revenue_per_patient = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .groupby('patient_id')['amount']
    .sum()
    .mean()
)

print(f"Average Revenue per Patient: ₹{average_revenue_per_patient:,.2f}")

Average Revenue per Patient: ₹5,100.73


## Top 10 Insurance Providers by Number of Patients.

In [62]:
top_insurance_providers = (
    patients_df
    .groupby('insurance_provider')
    .size()
    .reset_index(name='Total Patients')
    .sort_values(by='Total Patients', ascending=False)
    .head(10)
)

top_insurance_providers

,insurance_provider,Total Patients
1,MedCare Plus,18
3,WellnessCorp,16
2,PulseSecure,10
0,HealthIndia,6


## Monthly Trend of Reasons for Visit.

In [63]:
reason_visit_trend = (
    appointment_df
    .groupby([
        appointment_df['appointment_date'].dt.to_period('M'),
        'reason_for_visit'
    ])
    .size()
    .reset_index(name='Total Visits')
)

reason_visit_trend['appointment_date'] = reason_visit_trend['appointment_date'].astype(str)

reason_visit_trend

,appointment_date,reason_for_visit,Total Visits
0,2023-01,Checkup,3
1,2023-01,Consultation,3
2,2023-01,Emergency,4
3,2023-01,Follow-up,3
4,2023-01,Therapy,7
5,2023-02,Checkup,6
6,2023-02,Consultation,3
7,2023-02,Emergency,1
8,2023-02,Follow-up,2
9,2023-02,Therapy,2


## Monthly Revenue Trend by Payment Method

In [64]:
revenue_payment_trend = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .groupby([
        billing_df[billing_df['payment_status'] == 'Paid']['bill_date'].dt.to_period('M'),
        'payment_method'
    ])['amount']
    .sum()
    .reset_index()
)

revenue_payment_trend['bill_date'] = revenue_payment_trend['bill_date'].astype(str)

revenue_payment_trend

,bill_date,payment_method,amount
0,2023-01,Cash,6768.61
1,2023-01,Credit Card,9110.27
2,2023-01,Insurance,4201.76
3,2023-02,Credit Card,3032.56
4,2023-03,Cash,2057.45
5,2023-03,Credit Card,8734.27
6,2023-03,Insurance,8804.02
7,2023-04,Cash,5589.78
8,2023-04,Credit Card,3503.18
9,2023-04,Insurance,1286.77


## Monthly Revenue Trend by Hospital Branch

In [65]:
revenue_branch_trend = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .merge(
        treatments_df[['treatment_id', 'appointment_id']],
        on='treatment_id',
        how='left'
    )
    .merge(
        appointment_df[['appointment_id', 'doctor_id']],
        on='appointment_id',
        how='left'
    )
    .merge(
        doctors_df[['doctor_id', 'hospital_branch']],
        on='doctor_id',
        how='left'
    )
    .groupby([
        billing_df[billing_df['payment_status'] == 'Paid']['bill_date'].dt.to_period('M'),
        'hospital_branch'
    ])['amount']
    .sum()
    .reset_index()
)

revenue_branch_trend['bill_date'] = revenue_branch_trend['bill_date'].astype(str)

revenue_branch_trend

,bill_date,hospital_branch,amount
0,2023-01,Central Hospital,1096.36
1,2023-01,Eastside Clinic,10463.12
2,2023-03,Central Hospital,1315.17
3,2023-03,Westside Clinic,3628.15
4,2023-04,Westside Clinic,3228.14
5,2023-05,Central Hospital,7474.50
6,2023-06,Central Hospital,10311.47
7,2023-06,Eastside Clinic,6067.50
8,2023-07,Central Hospital,2212.80
9,2023-08,Westside Clinic,3349.18


## Monthly Appointment Trend by Doctor Specialization.

In [66]:
appointment_specialization_trend = (
    appointment_df
    .merge(
        doctors_df[['doctor_id', 'specialization']],
        on='doctor_id',
        how='left'
    )
    .groupby([
        appointment_df['appointment_date'].dt.to_period('M'),
        'specialization'
    ])
    .size()
    .reset_index(name='Appointments')
)

appointment_specialization_trend['appointment_date'] = (
    appointment_specialization_trend['appointment_date']
    .astype(str)
)

appointment_specialization_trend

,appointment_date,specialization,Appointments
0,2023-01,Dermatology,7
1,2023-01,Oncology,4
2,2023-01,Pediatrics,9
3,2023-02,Dermatology,6
4,2023-02,Oncology,2
5,2023-02,Pediatrics,6
6,2023-03,Dermatology,7
7,2023-03,Oncology,2
8,2023-03,Pediatrics,10
9,2023-04,Dermatology,7


## Monthly Treatment Trend by Treatment Type.

In [67]:
treatment_trend = (
    treatments_df
    .groupby([
        treatments_df['treatment_date'].dt.to_period('M'),
        'treatment_type'
    ])
    .size()
    .reset_index(name='Total Treatments')
)

treatment_trend['treatment_date'] = treatment_trend['treatment_date'].astype(str)

treatment_trend

,treatment_date,treatment_type,Total Treatments
0,2023-01,Chemotherapy,4
1,2023-01,ECG,4
2,2023-01,MRI,2
3,2023-01,Physiotherapy,4
4,2023-01,X-Ray,6
5,2023-02,Chemotherapy,3
6,2023-02,ECG,3
7,2023-02,MRI,2
8,2023-02,Physiotherapy,3
9,2023-02,X-Ray,3


## Monthly Revenue Trend by Treatment Type

In [68]:
revenue_treatment_trend = (
    billing_df[billing_df['payment_status'] == 'Paid']
    .merge(
        treatments_df[['treatment_id', 'treatment_type']],
        on='treatment_id',
        how='left'
    )
    .groupby([
        billing_df[billing_df['payment_status'] == 'Paid']['bill_date'].dt.to_period('M'),
        'treatment_type'
    ])['amount']
    .sum()
    .reset_index()
)

revenue_treatment_trend['bill_date'] = (
    revenue_treatment_trend['bill_date']
    .astype(str)
)

revenue_treatment_trend

,bill_date,treatment_type,amount
0,2023-01,Chemotherapy,6898.09
1,2023-01,MRI,4661.39
2,2023-03,Chemotherapy,1315.17
3,2023-03,X-Ray,3628.15
4,2023-04,Chemotherapy,3228.14
5,2023-05,MRI,4186.35
6,2023-05,X-Ray,3288.15
7,2023-06,ECG,1526.36
8,2023-06,MRI,3731.55
9,2023-06,Physiotherapy,11121.06


## Appointments by Day of Week.

In [69]:
appointments_by_day = (
    appointment_df
    .assign(day_name=appointment_df['appointment_date'].dt.day_name())
    .groupby('day_name')
    .size()
    .reindex([
        'Monday', 'Tuesday', 'Wednesday',
        'Thursday', 'Friday', 'Saturday', 'Sunday'
    ])
    .reset_index(name='Appointments')
)

appointments_by_day

,day_name,Appointments
0,Monday,26
1,Tuesday,37
2,Wednesday,37
3,Thursday,28
4,Friday,23
5,Saturday,23
6,Sunday,26


## Appointments by Hour

In [70]:
appointments_by_hour = (
    appointment_df
    .assign(hour=appointment_df['appointment_time'].apply(lambda x: x.hour))
    .groupby('hour')
    .size()
    .reset_index(name='Appointments')
    .sort_values('hour')
)

appointments_by_hour

,hour,Appointments
0,8,21
1,9,17
2,10,19
3,11,18
4,12,21
5,13,20
6,14,20
7,15,28
8,16,15
9,17,21


## Outstanding Revenue

In [71]:
outstanding_revenue = (
    billing_df[billing_df['payment_status'] != 'Paid']['amount']
    .sum()
)

print(f"Outstanding Revenue: ₹{outstanding_revenue:,.2f}")

Outstanding Revenue: ₹377,824.95


## Monthly Appointment Status Trend

In [72]:
appointment_status_trend = (
    appointment_df
    .groupby([
        appointment_df['appointment_date'].dt.to_period('M'),
        'status'
    ])
    .size()
    .reset_index(name='Appointments')
)

appointment_status_trend['appointment_date'] = (
    appointment_status_trend['appointment_date']
    .astype(str)
)

appointment_status_trend

,appointment_date,status,Appointments
0,2023-01,Cancelled,6
1,2023-01,Completed,1
2,2023-01,No-show,8
3,2023-01,Scheduled,5
4,2023-02,Cancelled,3
5,2023-02,Completed,6
6,2023-02,No-show,3
7,2023-02,Scheduled,2
8,2023-03,Cancelled,4
9,2023-03,Completed,4


## Save all your analysis DataFrames in one place so they're ready for the Streamlit dashboard.

In [74]:
analysis_data = {
    'gender_distribution': gender_distribution,
    'age_group_distribution': age_group_distribution,
    'registration_trend': registration_trend,
    'specialization_distribution': specialization_distribution,
    'branch_distribution': branch_distribution,
    'appointment_status': appointment_status,
    'monthly_appointments': monthly_appointments,
    'treatment_distribution': treatment_distribution,
    'payment_method_distribution': payment_method_distribution,
    'payment_status_distribution': payment_status_distribution,
    'monthly_revenue': monthly_revenue,
    'average_treatment_cost': average_treatment_cost,
    'top_10_paid_bills': top_10_paid_bills,
    'revenue_by_payment_method': revenue_by_payment_method,
    'revenue_by_treatment': revenue_by_treatment,
    'top_doctors': top_doctors,
    'top_patients': top_patients,
    'top_doctors_revenue': top_doctors_revenue,
    'revenue_by_branch': revenue_by_branch,
    'revenue_by_specialization': revenue_by_specialization,
    'top_insurance_providers': top_insurance_providers,
    'reason_visit_trend': reason_visit_trend,
    'revenue_payment_trend': revenue_payment_trend,
    'revenue_branch_trend': revenue_branch_trend,
    'appointment_specialization_trend': appointment_specialization_trend,
    'treatment_trend': treatment_trend,
    'revenue_treatment_trend': revenue_treatment_trend,
    'appointments_by_day': appointments_by_day,
    'appointments_by_hour': appointments_by_hour,
    'appointment_status_trend': appointment_status_trend
}